# 이진 분류 실험 ③ — 계좌 단기 활동 피처 추가 (account activity)
1단계 피처 사다리의 두 번째 칸. 02(기본 4 + port 2 + pair 4) 위에 **거래 직전 두 계좌의 활동량** 피처 8개를 더한다. 모델은 GINe 고정, 나머지 설정은 01·02 와 동일.

| 피처 | 뜻 (전부 그 거래 시각 기준 과거, 같은 분 제외) |
|---|---|
| `src_out_1h` / `src_out_1d` | 송신 계좌가 최근 60분 / 24시간 안에 보낸 거래 수 |
| `src_uniq_dst_1d` | 송신 계좌가 최근 24시간 안에 보낸 **상대 계좌 수** (일 단위 첫 거래 수로 근사) |
| `src_out_amt_1d` | 송신 계좌가 최근 24시간 안에 보낸 금액 합 |
| `dst_in_1h` / `dst_in_1d` | 수신 계좌가 최근 60분 / 24시간 안에 받은 거래 수 |
| `dst_uniq_src_1d` | 수신 계좌가 최근 24시간 안에 받은 **상대 계좌 수** |
| `dst_in_amt_1d` | 수신 계좌가 최근 24시간 안에 받은 금액 합 |

전부 `ln(1 + x)`. 창은 1h(버스트)와 1d(하루 활동) 두 개 — GFP 논문도 fan/degree 를 시간 창으로 센다 (`sampling/reference.md` 6절).

**가설**: 한 계좌가 짧은 시간에 여러 상대와 거래하는 **FAN-OUT / FAN-IN**, 모았다 뿌리는 **GATHER-SCATTER / SCATTER-GATHER** 의 recall 이 빡빡한 컷(0.1%, argmax)에서 오른다.
02 에서 가장 약했던 **고차수 수신자 [20-49]·[50-99] 구간**의 AP 가 오르는지도 본다 (`dst_in_1d` 가 "지금 이 계좌가 허브인가" 를 직접 말해 줌).

**메모리** (02 에서 두 번 죽고 고친 것): ① 피처 parquet 은 tx 올리기 전에 만든다 (DuckDB 12GB + tx 10GB 면 한도 초과) ② 피처 열은 float16 으로 붙이고 z-norm 결과도 float16 ③ 피처 행렬은 memmap 으로 두어 RSS 에서 뺀다. 셀마다 `B.trim` 으로 RSS 를 찍는다.

**누수 규칙**: 02 와 같음 — 거래 시각 기준 과거만, 같은 분 제외, z-norm 은 학습 구간 통계. 공통 코드 `bc_lib.py`

# 공통 부분 (작업 준비, 데이터 준비)

## 0. 작업 준비

In [1]:
import sys, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L

In [2]:
# ---- 실험 설정 ------------------------------------------------------------
RUN_NAME = 'account_activity'     # results/<DATASET>/account_activity/
PREV_RUN = 'pair_history'         # 비교 기준 (02)
cfg = B.Config(dataset='HI-Large', smoke=False)
B.seed_all(cfg.seed)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = cfg.out_dir(RUN_NAME)

# 앞 단계 피처 (02 에서 만든 parquet 재사용) + 이번 단계 피처
PAIR_FEATURES = ['pair_prior_count', 'pair_gap_minutes', 'pair_count_1h', 'reverse_history_count']
PAIR_PATH = cfg.feat_dir / 'pair_history.parquet'
STAGE_FEATURES = ['src_out_1h', 'src_out_1d', 'src_uniq_dst_1d', 'src_out_amt_1d', 'dst_in_1h', 'dst_in_1d', 'dst_uniq_src_1d', 'dst_in_amt_1d']
STAGE_PATH = cfg.feat_dir / 'account_activity.parquet'
GNN_EDGE_FEATURES = B.BASIC_EDGE_FEATURES + B.PORT_FEATURES + PAIR_FEATURES + STAGE_FEATURES   # FEAT16 열 순서. port 는 4,5 번
TREE_FEATURES = B.BASIC_EDGE_FEATURES + PAIR_FEATURES + STAGE_FEATURES                         # 트리는 port 없이
assert PAIR_PATH.exists(), '02 를 먼저 돌려 pair_history.parquet 을 만들어야 한다'
DEVICE, OUT_DIR, len(GNN_EDGE_FEATURES), len(TREE_FEATURES)

(device(type='cuda'),
 PosixPath('/workspace/money_laundry_ai_/code/binary_classification/results/HI-Large/account_activity'),
 18,
 16)

In [3]:
cfg

Config(dataset='HI-Large', smoke=False, val_neg_rate=0.02, scale_pos_weight=6, tree_rounds=500, tree_early_stop=20, num_neighbors=[100, 100], batch_size=2048, iter_per_epoch=256, epochs=400, eval_period=4, patience=8, class_weight=[1.0, 6.27], n_hidden=66, n_gnn_layers=2, lr=0.006, dropout=0.01, final_dropout=0.1, num_workers=0, seed=42)

## 1. 이번 단계 피처 계산 — 계좌 단기 활동 (tx 를 올리기 **전에**)
DuckDB 창 함수가 메모리 한도(12GB)까지 쓴다. 1.8억 건 `tx`(15열, 약 10GB)를 먼저 올려 두면 둘이 합쳐 25GB 를 넘겨 커널이 죽는다 → 피처 parquet 을 먼저 만들고 그 다음 tx 를 올린다 (한 번 만들면 이 절은 바로 지나간다)

- DuckDB 창 함수 두 번: **송신 계좌(src) 파티션** 한 번, **수신 계좌(dst) 파티션** 한 번. 02 의 pair 창과 같은 꼴 (`RANGE … PRECEDING AND INTERVAL '1 minute' PRECEDING`)
- 상대 계좌 수(`uniq_*_1d`): 창 함수로 `count(DISTINCT)` 를 1.8억 건에 돌리면 무거워서, **(계좌, 상대, 일) 의 첫 거래에 1 을 찍고 24시간 창에서 합산**으로 근사. 같은 상대가 어제 밤·오늘 아침 둘 다 있으면 2로 세는 오차가 있지만 방향은 같다
- 금액은 `received`(Amount Received, 기본 피처와 같은 열) 의 창 합계
- 결과는 `features_bc/account_activity.parquet` (edge_id 순). 한 번 만들면 재사용. pandas 를 거치지 않는다

In [4]:
con = L.connect(threads=8, memory='12GB')
currency_codes, format_codes = B.code_tables(con, cfg)
PORTS_PATH = B.ensure_ports(con, cfg)               # 있으면 건너뜀

In [5]:
# 이전 실행이 COPY 도중 죽었으면 불완전한 parquet 이 남아 있을 수 있다 → 행 수가 tx 와 다르면 지우고 다시 만든다
if STAGE_PATH.exists():
    n_stage = con.execute(f"SELECT count(*) FROM read_parquet({L.q(STAGE_PATH)})").fetchone()[0]
    n_tx = con.execute(f"SELECT count(*) FROM read_parquet({L.q(cfg.tx_path)})").fetchone()[0]
    if n_stage != n_tx:
        print(f'불완전한 {STAGE_PATH.name} ({n_stage:,} / {n_tx:,} 행) → 삭제 후 재생성'); STAGE_PATH.unlink()
if not STAGE_PATH.exists():
    t0 = time.time()
    con.execute(f"""
        CREATE OR REPLACE TABLE tx_min AS
        SELECT edge_id, src, dst, ts, day, received FROM read_parquet({L.q(cfg.tx_path)})""")
    # (a) 송신 계좌 기준 — 같은 src 의 과거 보낸 거래. first_pair_day = (src, dst, 일) 안의 첫 거래 표시 (상대 수 근사용)
    con.execute("""
        CREATE OR REPLACE TABLE src_win AS
        SELECT edge_id,
               count(*)            OVER w1h AS out_1h,
               count(*)            OVER w1d AS out_1d,
               sum(first_pair_day) OVER w1d AS uniq_dst_1d,
               sum(received)       OVER w1d AS out_amt_1d
        FROM (SELECT *, CASE WHEN row_number() OVER (PARTITION BY src, dst, day ORDER BY ts, edge_id) = 1 THEN 1 ELSE 0 END AS first_pair_day FROM tx_min)
        WINDOW w1h AS (PARTITION BY src ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING),
               w1d AS (PARTITION BY src ORDER BY ts RANGE BETWEEN INTERVAL '24 hours'  PRECEDING AND INTERVAL '1 minute' PRECEDING)""")
    print(f'  src_win {time.time()-t0:.0f}s')
    # (b) 수신 계좌 기준 — 같은 dst 의 과거 받은 거래
    con.execute("""
        CREATE OR REPLACE TABLE dst_win AS
        SELECT edge_id,
               count(*)            OVER w1h AS in_1h,
               count(*)            OVER w1d AS in_1d,
               sum(first_pair_day) OVER w1d AS uniq_src_1d,
               sum(received)       OVER w1d AS in_amt_1d
        FROM (SELECT *, CASE WHEN row_number() OVER (PARTITION BY dst, src, day ORDER BY ts, edge_id) = 1 THEN 1 ELSE 0 END AS first_pair_day FROM tx_min)
        WINDOW w1h AS (PARTITION BY dst ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING),
               w1d AS (PARTITION BY dst ORDER BY ts RANGE BETWEEN INTERVAL '24 hours'  PRECEDING AND INTERVAL '1 minute' PRECEDING)""")
    print(f'  dst_win {time.time()-t0:.0f}s')
    # (c) ln 변환 후 저장 (edge_id 순)
    con.execute(f"""
        COPY (
            SELECT t.edge_id,
                   ln(1 + s.out_1h)::FLOAT                   AS src_out_1h,
                   ln(1 + s.out_1d)::FLOAT                   AS src_out_1d,
                   ln(1 + coalesce(s.uniq_dst_1d, 0))::FLOAT AS src_uniq_dst_1d,
                   ln(1 + coalesce(s.out_amt_1d, 0))::FLOAT  AS src_out_amt_1d,
                   ln(1 + d.in_1h)::FLOAT                    AS dst_in_1h,
                   ln(1 + d.in_1d)::FLOAT                    AS dst_in_1d,
                   ln(1 + coalesce(d.uniq_src_1d, 0))::FLOAT AS dst_uniq_src_1d,
                   ln(1 + coalesce(d.in_amt_1d, 0))::FLOAT   AS dst_in_amt_1d
            FROM tx_min t JOIN src_win s USING (edge_id) JOIN dst_win d USING (edge_id)
            ORDER BY t.edge_id
        ) TO {L.q(STAGE_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    con.execute("DROP TABLE src_win; DROP TABLE dst_win; DROP TABLE tx_min")
    print(f'{STAGE_PATH.name} 생성 {time.time()-t0:.0f}s')
con.execute(f"SELECT count(*), {', '.join(f'avg({c}) AS avg_{c}' for c in STAGE_FEATURES)} FROM read_parquet({L.q(STAGE_PATH)})").df().T

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,0
count_star(),1.797021e+08
avg_src_out_1h,1.063768e+00
avg_src_out_1d,2.236039e+00
avg_src_uniq_dst_1d,1.825606e+00
avg_src_out_amt_1d,9.582797e+00
avg_dst_in_1h,4.394570e-01
avg_dst_in_1d,1.075625e+00
avg_dst_uniq_src_1d,7.866559e-01
avg_dst_in_amt_1d,7.244756e+00


In [6]:
# 창 동작 확인 — 하루에 거래 8건쯤 보낸 계좌 하나를 골라 그 계좌의 보낸 거래 타임라인 (out_1h 는 1시간 지나면 줄고, out_1d 는 하루 안에서 1씩 늘고, uniq 는 새 상대일 때만 느는지)
# 주의: 1.8억 건끼리 조인하지 않는다 → 그 계좌의 edge_id 만 골라 피처 parquet 에서 읽는다
acct = con.execute(f"SELECT src FROM read_parquet({L.q(cfg.tx_path)}) GROUP BY src, day ORDER BY abs(count(*) - 8), src LIMIT 1").fetchone()[0]
ids = con.execute(f"SELECT edge_id, ts, dst, received FROM read_parquet({L.q(cfg.tx_path)}) WHERE src = {acct} ORDER BY ts LIMIT 40").df()
feat = con.execute(f"SELECT edge_id, src_out_1h, src_out_1d, src_uniq_dst_1d, src_out_amt_1d FROM read_parquet({L.q(STAGE_PATH)}) WHERE edge_id IN ({', '.join(map(str, ids.edge_id))})").df()
tl = ids.merge(feat, on='edge_id').sort_values('ts')
for c in ['src_out_1h', 'src_out_1d', 'src_uniq_dst_1d']:
    tl[c.replace('src_', '') + ' (건수)'] = np.expm1(tl[c]).round(0).astype(int)      # ln(1+x) 를 되돌려 건수로
tl

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,edge_id,ts,dst,received,src_out_1h,src_out_1d,src_uniq_dst_1d,src_out_amt_1d,out_1h (건수),out_1d (건수),uniq_dst_1d (건수)
0,130498,2022-08-01 00:03:00,0,11.21,0.000000,0.000000,0.000000,0.000000,0,0,0
1,1736297,2022-08-01 03:39:00,86029,521.90,0.000000,0.693147,0.693147,2.502255,0,1,1
2,1760854,2022-08-01 03:50:00,86029,564.35,0.693147,1.098612,1.098612,6.280602,1,2,2
3,2130513,2022-08-01 06:36:00,65675,1847.03,0.000000,1.386294,1.098612,7.001665,0,3,2
4,2179612,2022-08-01 06:58:00,65675,2285.52,0.693147,1.609438,1.386294,7.988030,1,4,3
5,3559332,2022-08-01 17:15:00,0,1259.35,0.000000,1.791759,1.386294,8.562360,0,5,3
6,3785254,2022-08-01 18:56:00,29873,1127.66,0.000000,1.945910,1.386294,8.778073,0,6,3
7,4269502,2022-08-01 22:32:00,0,65.15,0.000000,2.079442,1.609438,8.938272,0,7,4
8,4296345,2022-08-01 22:44:00,0,579.25,0.693147,2.197225,1.609438,8.946788,1,8,4
9,4750022,2022-08-02 02:52:00,29873,1127.66,0.000000,2.197225,1.386294,9.018115,0,8,3


## 2. 데이터 준비 — 거래 전체 + port + pair(02) + 이번 피처
전부 edge_id 순 parquet 이라 조인 없이 열만 붙인다 (`attach_parquet`)

In [8]:
con.close(); con = L.connect(threads=8, memory='12GB')   # 창 함수가 잡아 둔 DuckDB 버퍼를 돌려주고 새로 연결
con.register('currency_codes', currency_codes); con.register('format_codes', format_codes)   # 코드표는 연결마다 다시 등록
tx = B.load_tx(con, cfg)
tx = B.attach_parquet(tx, PORTS_PATH, B.PORT_FEATURES)
tx = B.attach_parquet(tx, PAIR_PATH, PAIR_FEATURES, dtype=np.float16)     # 02 의 pair 피처 — float16 으로 붙여 반으로 (ln 값이라 충분)
tx = B.attach_parquet(tx, STAGE_PATH, STAGE_FEATURES, dtype=np.float16)   # 이번 피처
B.trim('(tx 적재 후)'); tx.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

RSS 25.5 GB (tx 적재 후)


(179702079, 23)

In [9]:
tx.head()

/usr/local/lib/python3.12/dist-packages/pandas/io/formats/format.py:1466: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,edge_id,src,dst,day,timestamp,amount_received,received_currency,payment_format,y,in_port,...,pair_count_1h,reverse_history_count,src_out_1h,src_out_1d,src_uniq_dst_1d,src_out_amt_1d,dst_in_1h,dst_in_1d,dst_uniq_src_1d,dst_in_amt_1d
0,1,3,3,2022-08-01,0.0,19.180000,12,5,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,25,78803,2022-08-01,0.0,17955.109375,12,3,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,31,86028,2022-08-01,0.0,1199.829956,12,2,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,39,269980,2022-08-01,0.0,148565.453125,12,6,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,50,29866,2022-08-01,0.0,227247.421875,12,3,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
tx[STAGE_FEATURES].describe().T

/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1524: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:725: RuntimeWarning: invalid value encountered in scalar divide
  the_mean = the_sum / count if count > 0 else np.nan
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1524: RuntimeWarning: overflow encountered in cast
  return dtype.type(n)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:49: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:725: RuntimeWarning: invalid value encountered in scalar divide
  the_mean = the_sum / count if count > 0 else np.nan
/usr/local

,count,mean,std,min,25%,50%,75%,max
src_out_1h,179702079.0,NaN,0.0,0.0,0.000000,0.693359,1.098633,9.921875
src_out_1d,179702079.0,NaN,0.0,0.0,0.693359,1.609375,2.302734,11.687500
src_uniq_dst_1d,179702079.0,NaN,0.0,0.0,0.693359,1.098633,1.609375,10.929688
src_out_amt_1d,179702079.0,NaN,0.0,0.0,6.183594,9.414062,12.632812,30.875000
dst_in_1h,179702079.0,NaN,0.0,0.0,0.000000,0.000000,0.693359,4.875000
dst_in_1d,179702079.0,NaN,0.0,0.0,0.693359,1.098633,1.609375,7.742188
dst_uniq_src_1d,179702079.0,NaN,0.0,0.0,0.693359,0.693359,1.098633,7.742188
dst_in_amt_1d,179702079.0,NaN,0.0,0.0,4.632812,7.871094,10.320312,30.187500


In [11]:
# 라벨별 평균 — 세탁 거래에서 값이 다른가 (피처가 신호를 갖는지 미리 확인). pandas groupby 는 열 단위라 가볍다
tx.groupby('y')[STAGE_FEATURES].mean().T.rename(columns={0: 'normal', 1: 'laundering'})

y,normal,laundering
src_out_1h,1.064207,0.776982
src_out_1d,2.236996,1.593625
src_uniq_dst_1d,1.826310,1.378656
src_out_amt_1d,9.587427,5.898304
dst_in_1h,0.439792,0.244257
dst_in_1d,1.076208,0.739540
dst_uniq_src_1d,0.787036,0.587356
dst_in_amt_1d,7.247368,5.165527


## 3. z-norm (학습 구간 통계) + 표본

In [1]:
in_train = B.window_mask(tx, cfg.train_window)
feat_z, znorm_table = B.znorm_features(tx, GNN_EDGE_FEATURES, in_train)     # float16 행렬 (18열 6.5GB). tx 에서 피처 열은 떨어짐
B.trim('(z-norm 후)'); znorm_table

NameError: name 'B' is not defined

In [ ]:
print(f"데이터 기간 {tx['day'].min().date()} ~ {tx['day'].max().date()}")
for name, w in [('train', cfg.train_window), ('val', cfg.val_window), ('test', cfg.test_window)]:
    m = B.window_mask(tx, w); assert m.sum() > 0, f'{name} 구간 {w} 에 거래가 없음'
    print(f'{name:5s} {w}  거래 {m.sum():>12,}  세탁 {tx.loc[m, "y"].sum():>8,}  비율 {tx.loc[m, "y"].mean()*100:.3f}%')

In [ ]:
val_df = B.val_sample(con, cfg.val_window, cfg.val_neg_rate)        # 검증: 가중 표본 (05 공유)
test_df = B.test_full(con, cfg.test_window)                         # 테스트: 구간 전체
in_degree = B.recipient_indegree(con, cfg)                           # 분석용
con.close()

all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
assert (tx['y'].to_numpy()[val_idx] == val_df['y'].to_numpy()).all()
del all_edge_ids
val_df.shape, test_df.shape

# 트리 모델 (기본 4 + pair 4 + 이번 피처 8)

## 4. LightGBM / XGBoost

In [ ]:
results = {}
tree_cols = [GNN_EDGE_FEATURES.index(c) for c in TREE_FEATURES]
train_mask = in_train
X_train = feat_z[train_mask][:, tree_cols].astype(np.float32); y_train = tx['y'].to_numpy()[train_mask]   # 트리는 float32 (LightGBM 이 내부에서 어차피 변환)
X_val, y_val, w_val = feat_z[val_idx][:, tree_cols].astype(np.float32), val_df['y'].to_numpy(), val_df['w'].to_numpy()
X_test = feat_z[test_idx][:, tree_cols].astype(np.float32)
if cfg.smoke:
    keep = np.random.default_rng(cfg.seed).choice(len(X_train), min(len(X_train), 200_000), replace=False)
    X_train, y_train = X_train[keep], y_train[keep]
B.trim('(트리 입력 준비 후)'); X_train.shape, X_test.shape

In [ ]:
lgb_model, xgb_model = B.train_trees(X_train, y_train, X_val, y_val, w_val, X_test, val_df, test_df, cfg, results)
pd.DataFrame({'feature': TREE_FEATURES, 'lightgbm_gain': lgb_model.booster_.feature_importance('gain'),
              'xgboost_gain': [xgb_model.get_booster().get_score(importance_type='gain').get(f'f{i}', 0.0) for i in range(len(TREE_FEATURES))]})

In [ ]:
# 트리 테스트 점수도 저장 (분석용)
for name, model in [('lightgbm', lgb_model), ('xgboost', xgb_model)]:
    pd.DataFrame(dict(edge_id=test_df.edge_id, y=test_df.y, score=model.predict_proba(X_test)[:, 1])).to_parquet(OUT_DIR / f'{name}_test_scores.parquet', index=False)
del X_train, y_train, X_test; gc.collect()
pd.DataFrame({k: v['val'] for k, v in results.items()}).T.round(4)

# GNN (GINe, 01 과 같은 설정)

## 5. 그래프 — 한 번에 하나만 (bc_lib.GraphStore)

In [ ]:
# 피처 행렬은 memmap 으로 — 6.5GB 가 RSS 에서 빠진다 (페이지 캐시는 cgroup 이 회수 가능). 02 는 이 시점 RSS 약 13GB, 학습 중 20.7GB 였다
store = B.GraphStore(tx, feat_z, cfg, train_rows=np.flatnonzero(train_mask), val_idx=val_idx, test_idx=test_idx,
                     port_cols=[GNN_EDGE_FEATURES.index('in_port'), GNN_EDGE_FEATURES.index('out_port')],
                     feat_memmap=cfg.feat_dir / f'feat16_{RUN_NAME}.npy')
del tx, feat_z; B.trim('(tx, feat_z 삭제 후)')
store.edge_dim

In [ ]:
# 배치 하나 꺼내서 모양 확인 — 2048 target 이 서브그래프 하나를 공유, 엣지 피처 18열
batch = next(iter(store.use('train')))
print('노드', batch['node'].num_nodes, ' to 간선', batch[B.TO].num_edges, ' rev_to 간선', batch[B.REV].num_edges,
      ' target', batch[B.TO].edge_label_index.shape[1], ' 양성', int(batch[B.TO].edge_label.sum()),
      ' x 열', batch['node'].x.shape[1], ' edge_attr 열', batch[B.TO].edge_attr.shape[1], ' target_attr', tuple(batch.target_attr.shape))

In [ ]:
# 모델 모양 확인
model = B.make_gine(store.edge_dim, cfg, edge_updates=False).to(DEVICE)
logits, labels = B.forward_batch(model, batch, DEVICE)
print('파라미터', sum(p.numel() for p in model.parameters()), ' 로짓', tuple(logits.shape))
del model, batch

## 6. 학습 — GINe

In [ ]:
gine_model, gine_hist = B.train_gnn('gine', store, cfg, val_df, OUT_DIR, DEVICE, edge_updates=False, results=results)
results['gine']['val']

In [ ]:
# 학습 곡선
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
hv = gine_hist.dropna(subset=['AP']); ax[0].plot(hv['epoch'], hv['AP'], marker='o'); ax[1].plot(gine_hist['epoch'], gine_hist['train_loss'])
ax[0].set_title('val AP'); ax[1].set_title('train loss'); ax[0].set_xlabel('epoch'); ax[1].set_xlabel('epoch')
plt.tight_layout(); plt.savefig(OUT_DIR / 'curves.png', dpi=120); plt.show()

## 7. 테스트 — 구간 전체

In [ ]:
gine_score = B.test_gnn('gine', store, cfg, test_df, OUT_DIR, DEVICE, edge_updates=False, results=results)

# 결과 — 저장된 점수 파일만으로 돌아간다
커널을 새로 켰으면 **0 절(설정 셀 3개)까지만** 돌리고 바로 여기부터. 재료는 `results/<DATASET>/<RUN_NAME>/`
- `{model}_test_scores.parquet` (edge_id, y, score) — 7 절·4 절에서 저장
- `results.json` — 검증/테스트 지표 (`results` dict 복구용)
- dst 와 수신자 in-degree 만 DuckDB 에서 다시 읽는다 (1~2 분)

운영점(컷)은 두 종류: **예산 컷** `r@b%` = 테스트 전체 상위 b% 를 알림으로 (0.05 / 0.1 / 0.5 / 1 / 2 / 5 / 10 % / all),
**argmax** = 확률 0.5 (논문 F1 운영점, Multi-GNN 부록 Table 13 의 패턴별 recall 과 같은 기준). 0.1% ≈ 양성 수 규모 ≈ argmax 와 비슷한 알림 수.

## 8. 준비 — 점수 · 라벨 · in-degree

In [ ]:
import importlib; importlib.reload(B)
MODELS = [n for n in ['lightgbm', 'xgboost', 'gine'] if (OUT_DIR / f'{n}_test_scores.parquet').exists()]
scores = {n: B.load_scores(OUT_DIR / f'{n}_test_scores.parquet') for n in MODELS}
if 'test_df' not in globals() or 'in_degree' not in globals():
    con = L.connect(threads=8, memory='12GB')
    test_df = B.test_full(con, cfg.test_window); in_degree = B.recipient_indegree(con, cfg); con.close()
for n in MODELS:
    assert (scores[n]['edge_id'].to_numpy() == test_df['edge_id'].to_numpy()).all(), f'{n} 점수와 라벨 순서 불일치'
if 'results' not in globals() or not results:
    results = json.load(open(OUT_DIR / 'results.json'))['results']
print(MODELS, '| 테스트 간선', f"{len(test_df):,}", '| 양성', f"{int(test_df.y.sum()):,}", '| 컷', list(B.cut_thresholds(scores[MODELS[0]]['score'].to_numpy())))

## 9. 운영점별 precision / recall
같은 점수를 컷만 바꿔 가며 본다. 예산을 줄이면 precision 이 오르고 recall 이 떨어지는 그 교환을 한 표에.

In [ ]:
# 컷마다 임계값 · 알림 수 · TP/FP/FN · precision · recall (모델별). 'all' = 전부 알림 (precision = 양성 비율)
cut_df = pd.concat([B.cut_table(scores[n]).assign(model=n) for n in MODELS])
cut_df.to_csv(OUT_DIR / 'analysis_cuts.csv', encoding='utf-8-sig')
cut_df[cut_df.model == 'gine']

In [ ]:
# 모델 나란히 — 컷별 recall / precision
cut_df.reset_index().pivot(index='cut', columns='model', values=['recall', 'precision']).loc[cut_df.index.unique()]

In [ ]:
# PR 곡선 — 컷을 연속으로 훑은 것. AP = 곡선 아래 면적. 세로선 = GINe argmax 운영점
B.plot_pr(scores, path=OUT_DIR / 'pr_curve.png', mark=(cut_df.loc['argmax'].query("model == 'gine'")['recall'].iloc[0], 'GINe argmax'))

In [ ]:
# 혼동행렬 — GINe, 컷 4개 나란히. 색 = 행 비율 (세탁 행 오른쪽 칸 = recall), 숫자 = 건수
B.plot_confusion_grid(scores['gine'], cuts=('r@0.1%', 'r@1%', 'r@10%', 'argmax'), title='GINe (pair_history)', path=OUT_DIR / 'confusion_cuts.png')

## 10. 패턴별 — 어느 패턴을 어느 컷에서 놓치나
패턴 안의 거래는 모두 양성이라 recall 만 본다. 예산을 줄일 때 먼저 떨어지는 패턴이 "모델이 덜 확신하는" 패턴.

In [ ]:
# 패턴 × 컷 recall (모델별 저장, GINe 표시). 'recall' 열 = 1% (01 과 호환), mean_score = 그 패턴 양성의 평균 점수
pattern_df = pd.concat([B.pattern_recall(scores[n]).assign(model=n) for n in MODELS])
pattern_df.to_csv(OUT_DIR / 'analysis_pattern.csv', encoding='utf-8-sig')
CUT_COLS = [c for c in pattern_df.columns if c.startswith('r@') or c in ('all', 'argmax')]
pattern_df[pattern_df.model == 'gine'][['n'] + CUT_COLS + ['mean_score']]

In [ ]:
# 같은 표를 그림으로 — 행 = 패턴, 열 = 컷. 왼쪽(빡빡한 예산)에서 먼저 하얘지는 행이 약한 패턴
B.plot_heatmap(pattern_df[pattern_df.model == 'gine'][CUT_COLS], title='GINe recall: pattern x cut', path=OUT_DIR / 'pattern_x_cut.png')

In [ ]:
# 놓친 세탁(FN) 의 구성 — 컷마다 FN 이 어느 패턴에서 나오나. FN% 가 pos_share 보다 크면 그 패턴을 유독 놓침
groups = dict(pattern=B.pattern_array(scores['gine']), degree_bin=B.degree_bin_of(test_df['dst'].to_numpy(), in_degree))
cb = B.confusion_breakdown(scores['gine'], groups)
for axis in groups:
    cb['fn'][axis].to_csv(OUT_DIR / f'analysis_fn_{axis}.csv', encoding='utf-8-sig'); cb['fp'][axis].to_csv(OUT_DIR / f'analysis_fp_{axis}.csv', encoding='utf-8-sig')
cb['fn']['pattern'][['pos_share'] + [c for c in cb['fn']['pattern'].columns if c.startswith('FN% ')]]

## 11. 수신자 in-degree 층화 (SALT-GNN 프로토콜)
허브(차수 큰 수신자) 쪽 거래가 약점인지. AP/F1 은 구간 안에서 따로 계산, recall/precision 은 전체 컷 임계값을 구간에 적용.

In [ ]:
# 구간별 AP · F1 · 양성 비율 (모델별 저장, GINe 표시)
degree_df = pd.concat([B.degree_stratified(scores[n], test_df['dst'].to_numpy(), in_degree).assign(model=n) for n in MODELS])
degree_df.to_csv(OUT_DIR / 'analysis_degree.csv', encoding='utf-8-sig')
degree_df[degree_df.model == 'gine']

In [ ]:
# 구간 × 컷 — recall 과 precision (GINe). 같은 임계값인데 구간마다 precision 이 다르면 알림이 그쪽에 몰리는 것
deg_cut = B.group_by_cut(scores['gine'], groups['degree_bin'])
order = [b for b in [f'[{lo}-{hi}]' if hi < 10**9 else f'[{lo}+]' for lo, hi in B.DEGREE_BINS] if b in deg_cut['recall'].index]
deg_cut['recall'].loc[order].to_csv(OUT_DIR / 'analysis_degree_x_cut_recall.csv', encoding='utf-8-sig')
deg_cut['precision'].loc[order].to_csv(OUT_DIR / 'analysis_degree_x_cut_precision.csv', encoding='utf-8-sig')
display(deg_cut['recall'].loc[order]); deg_cut['precision'].loc[order]

In [ ]:
# 잘못 잡은 정상(FP) 이 어느 차수 구간에 몰리나 — FP% 가 neg_share 보다 크면 그 구간에 알림이 과함
cb['fp']['degree_bin'][['neg_share'] + [c for c in cb['fp']['degree_bin'].columns if c.startswith('FP% ')]]

## 12. 앞 단계 (PREV_RUN) 와 비교
앞 단계 결과 폴더의 `analysis_*.csv` 와 비교. 01 은 모델 이름이 `gine_basic`, 02 부터는 `gine`.

In [ ]:
# 패턴별 recall — 1% 와 argmax 두 컷에서 앞 단계 GINe vs 이번 GINe
PREV_GNN = 'gine_basic' if PREV_RUN == 'basic_features' else 'gine'
prev_path = Path.cwd() / 'results' / cfg.dataset / PREV_RUN / 'analysis_pattern.csv'
if prev_path.exists():
    prev = pd.read_csv(prev_path, index_col=0); prev = prev[prev.model == PREV_GNN]; cur = pattern_df[pattern_df.model == 'gine']
    cmp = pd.DataFrame({f'{PREV_RUN} r@1%': prev['recall'], f'{RUN_NAME} r@1%': cur['recall']})
    cmp['diff r@1%'] = cmp.iloc[:, 1] - cmp.iloc[:, 0]
    if 'argmax' in prev.columns:
        cmp[f'{PREV_RUN} argmax'] = prev['argmax']; cmp[f'{RUN_NAME} argmax'] = cur['argmax']; cmp['diff argmax'] = cur['argmax'] - prev['argmax']
    display(cmp.loc[[p for p in B.PATTERN_ORDER + ['ALL'] if p in cmp.index]].round(4))
else:
    print('앞 단계 분석표 없음 —', prev_path)

In [ ]:
# 수신 계좌 degree 구간별 AP — 허브 구간이 달라졌나
prev_path = Path.cwd() / 'results' / cfg.dataset / PREV_RUN / 'analysis_degree.csv'
if prev_path.exists():
    prev_d = pd.read_csv(prev_path, index_col=0); prev_d = prev_d[prev_d.model == PREV_GNN]; cur_d = degree_df[degree_df.model == 'gine']
    display(pd.DataFrame({f'{PREV_RUN} AP': prev_d['AP'], f'{RUN_NAME} AP': cur_d['AP'], 'diff': cur_d['AP'] - prev_d['AP'],
                          f'{PREV_RUN} F1': prev_d['F1'], f'{RUN_NAME} F1': cur_d['F1'], 'n_pos': cur_d['n_pos']}).round(4))
else:
    print('앞 단계 분석표 없음:', prev_path)

## 13. 비교표 + 저장

In [ ]:
compare_df = B.save_results(OUT_DIR, cfg, results, features=dict(tree=TREE_FEATURES, gnn=GNN_EDGE_FEATURES))
compare_df

In [ ]:
# 앞 단계 전체 지표와 나란히
prev_c = pd.read_csv(Path.cwd() / 'results' / cfg.dataset / PREV_RUN / 'compare.csv', index_col=[0, 1])
pd.concat([prev_c.assign(run=PREV_RUN), compare_df.assign(run=RUN_NAME)]).reset_index().set_index(['run', 'model', 'split'])[['AP', 'F1', 'F1_at_0.5', 'recall_at_1pct', 'unmatched_recall_at_1pct']]

In [ ]:
sorted(p.name for p in OUT_DIR.iterdir())   # 이 폴더가 report.md 의 재료

## 14. 다음
- 04: 계좌 금액 분포 (1일 창 mean/std/max, 이 거래 금액이 그 계좌 평소 대비 몇 배) → 스머핑·허브 구별
- 05: GFP 모티프 (사이클·scatter-gather 참여 수, snapml GraphFeaturePreprocessor) → CYCLE, SCATTER-GATHER
- 그 다음 v10 36피처 참조 런 → 1단계 정리